# Can we predict `failed_90d` after an inspection?

Plan (per checklist.md):
1. Sort by `inspected_at`; the latest 20% of rows is the test set (the model is meant for future inspections). Every `asset_tag` appears once, so there is no entity overlap to handle.
2. Time-ordered cross-validation (expanding window) inside the training rows only.
3. Preprocessing lives inside sklearn pipelines, so it is fit within each fold.
4. Candidates are named before any scoring.
5. Model and threshold are chosen on CV only, and written down before the test set is touched.
6. Refit on all training rows, score the test set once.
7. `parts_replaced` is recorded on the work order that *closes* the inspection record, i.e. after the inspection, so it is not known at prediction time. It is dropped (it is also near-perfectly tied to the label: 2+ parts means every asset failed). `asset_tag` is an identifier and is dropped.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support

SEED = 0
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'positive rate', round(df.failed_90d.mean(), 3), 'unique assets', df.asset_tag.nunique())
print(df.inspected_at.min(), '->', df.inspected_at.max())

## 1. Test split (latest rows)

Note: rows inspected within 90 days of the last date in the data may have an incomplete outcome window; this affects the latest rows and is a caveat for the test set.

In [ ]:
n_test = int(len(df) * 0.2)
train, test = df.iloc[:-n_test].copy(), df.iloc[-n_test:].copy()
print('train', train.inspected_at.min(), train.inspected_at.max(), len(train))
print('test ', test.inspected_at.min(), test.inspected_at.max(), len(test))
print('train/test positive rate', train.failed_90d.mean().round(3), test.failed_90d.mean().round(3))

## 2. Features and candidate models (named before scoring)

Features: `vibration_rms`, `bearing_temp_c`, `ambient_temp_c`, `meter_hours`, `load_pct`, `site`.

Candidates:
- **baseline**: predicts the training prior (no skill, AUC 0.5)
- **logreg**: standardised numerics + one-hot site, logistic regression
- **hgb**: histogram gradient boosting, depth 3, lr 0.05, 150 iterations

Primary metric: average precision (positives are ~12%); ROC AUC is reported as well.

In [ ]:
NUM = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
FEATURES = NUM + ['site']
def prep():
    return ColumnTransformer([('n', StandardScaler(), NUM), ('c', OneHotEncoder(handle_unknown='ignore'), ['site'])])
candidates = {
    'baseline': lambda: DummyClassifier(strategy='prior'),
    'logreg': lambda: make_pipeline(prep(), LogisticRegression(max_iter=1000)),
    'hgb': lambda: make_pipeline(prep(), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=SEED)),
}

## 3. Time-ordered cross-validation on training rows only

In [ ]:
X, y = train[FEATURES], train.failed_90d
cv = TimeSeriesSplit(n_splits=5)
oof = {k: np.full(len(train), np.nan) for k in candidates}
rows = []
for name, make in candidates.items():
    aucs, aps = [], []
    for tr_i, va_i in cv.split(X):
        m = make().fit(X.iloc[tr_i], y.iloc[tr_i])
        p = m.predict_proba(X.iloc[va_i])[:, 1]
        oof[name][va_i] = p
        aucs.append(roc_auc_score(y.iloc[va_i], p)); aps.append(average_precision_score(y.iloc[va_i], p))
    rows.append((name, np.mean(aucs), np.std(aucs), np.mean(aps), np.std(aps)))
cv_table = pd.DataFrame(rows, columns=['model', 'auc_mean', 'auc_sd', 'ap_mean', 'ap_sd']).round(3)
cv_table

## 4. Choose model and decision threshold on CV out-of-fold predictions

In [ ]:
best = cv_table.sort_values('ap_mean', ascending=False).iloc[0].model
grid = np.linspace(0.02, 0.8, 79)
fold_thr = []
for tr_i, va_i in cv.split(X):
    p = oof[best][va_i]
    fold_f1 = [precision_recall_fscore_support(y.iloc[va_i], p >= t, average='binary', zero_division=0)[2] for t in grid]
    fold_thr.append(grid[int(np.argmax(fold_f1))])
threshold = float(np.median(fold_thr))
print('chosen model:', best, '| per-fold best thresholds:', np.round(fold_thr, 2), '| threshold (median):', round(threshold, 2))

## Choice (recorded before scoring the test set)
Selected on time-ordered CV only: **hgb** (CV AP 0.369 vs logreg 0.344 vs baseline 0.162; CV AUC 0.755), features `vibration_rms, bearing_temp_c, ambient_temp_c, meter_hours, load_pct, site`, hyperparameters as listed above (not tuned), decision threshold **0.13** (median of the per-fold F1-optimal thresholds, which are noisy: 0.02 to 0.54). CV scores are noisy (AP sd ~0.14 across folds) and the label rate varies strongly by season (about 5% in some months, over 35% in summer), so models are only modestly separated. The test set was not used for any of these decisions.

## 5. Refit on all training rows; score the test set once

In [ ]:
final = candidates[best]().fit(X, y)
p_test = final.predict_proba(test[FEATURES])[:, 1]
yt = test.failed_90d
prec, rec, f, _ = precision_recall_fscore_support(yt, p_test >= threshold, average='binary', zero_division=0)
print(f'TEST  AUC {roc_auc_score(yt, p_test):.3f} | AP {average_precision_score(yt, p_test):.3f} | '
      f'precision {prec:.3f} recall {rec:.3f} F1 {f:.3f} at threshold {threshold:.2f}')
print('reference: test positive rate (AP of a no-skill model) =', round(yt.mean(), 3))

## Conclusion
See the headline test numbers above (single scoring, no re-tuning afterwards).